In [1]:
# ============================================================
# Cosmic-Net: Symbolic Regression — FIXED + EXTENDED
# Uses only leak-free features as input
# Target: log(M_halo) — using correct independent features only
# Physical constants included for equation discovery
# Runtime: ~6-7 hours on Kaggle CPU (pop=15000, gen=300)
# ============================================================

import subprocess, sys, os, json, warnings, time
warnings.filterwarnings('ignore')

subprocess.run([sys.executable, '-m', 'pip', 'install', 'gplearn', 'sympy', '-q'],
               check=True)
print("Dependencies installed")

import numpy as np
import pandas as pd
from gplearn.genetic import SymbolicRegressor
from gplearn.functions import make_function
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.decomposition import PCA

# ── Physical Constants (log10 scale) ─────────────────────────
# These become candidate terminals in the GP search
# G in units of Mpc (km/s)^2 / M_sun
LOG_G       = np.log10(4.302e-9)   # -8.366
LOG_PI      = np.log10(np.pi)      # 0.497
LOG_E       = np.log10(np.e)       # 0.434
LOG_C       = np.log10(2.998e5)    # 5.477  (speed of light in km/s)
LOG_HBAR    = np.log10(1.055e-34)  # -33.977 (Planck/2pi, SI)
LOG_H100    = np.log10(0.6774)     # -0.169  (Hubble parameter h)

print(f"\nPhysical constants (log10):")
print(f"  log10(G)    = {LOG_G:.4f}   [Mpc (km/s)^2 / M_sun]")
print(f"  log10(pi)   = {LOG_PI:.4f}")
print(f"  log10(e)    = {LOG_E:.4f}")
print(f"  log10(c)    = {LOG_C:.4f}   [km/s]")
print(f"  log10(h)    = {LOG_H100:.4f}  [Hubble param]")

# ── Load Data (OLD TNG DATA) ──────────────────────────────────
print("\nLoading data...")
df = pd.read_csv(
    '/kaggle/input/datasets/rusheelsharma/tng-fixed3/tng100_clustered.csv'
)

# ── Data Preparation (UPDATED for Corrected Data) ────────────
# We no longer skip the central galaxy. Its mass is the best predictor 
# and is no longer "leaking" the target value.

# Ensure linear scale for satellite aggregations
df['stellar_mass_linear'] = 10 ** df['stellar_mass']

# Unit convert positions/radii if necessary
H = 0.6774
CKPC_TO_MPC = 1e-3 / H
if df['half_mass_radius'].median() > 1.0:
    df['half_mass_radius_mpc'] = df['half_mass_radius'] * CKPC_TO_MPC
else:
    df['half_mass_radius_mpc'] = df['half_mass_radius']

print("\nProcessing Centrals and Satellites...")
centrals   = df[df['is_central'] == 1].copy()
satellites = df[df['is_central'] == 0].copy()

# Target and Central Features
cen_agg = centrals[['group_id', 'halo_mass_log', 'stellar_mass', 
                    'vel_dispersion', 'half_mass_radius_mpc', 
                    'metallicity']].copy()
cen_agg.columns = ['group_id', 'halo_mass_log', 'log_M_cen', 
                   'cen_vel_disp', 'cen_half_mass_r', 'cen_metallicity']

# Satellite Features
sat_agg = satellites.groupby('group_id').agg(
    n_satellites        = ('subhalo_id', 'count'),
    mean_vel_disp       = ('vel_dispersion', 'mean'),
    total_sat_mass      = ('stellar_mass_linear', 'sum'),
    mean_half_mass_r    = ('half_mass_radius_mpc', 'mean')
).reset_index()

# Merge
halo = cen_agg.merge(sat_agg, on='group_id', how='left')
halo.fillna(0, inplace=True) 

y_old = halo['halo_mass_log'].values

# Feature Engineering
def safe_log(x):
    return np.log10(np.clip(np.abs(x), 1e-10, None))

log_M_cen       = halo['log_M_cen'].values
log_sigma_cen   = safe_log(halo['cen_vel_disp'])
log_R_cen       = safe_log(halo['cen_half_mass_r'])
log_sigma_sat   = safe_log(halo['mean_vel_disp'])
log_N_sat       = safe_log(halo['n_satellites'])
log_M_sat_total = safe_log(halo['total_sat_mass'])

# ── Load Data (NEW GNN DATA) ──────────────────────────────────
print("\nLoading GNN Embedding data...")
df_gnn = pd.read_csv('/kaggle/input/datasets/rusheelsharma/gnn-embed/gnn_embeddings_for_sr.csv')

# 1. Robust Target Identification (Fixes the KeyError)
potential_targets = ['target_halo_mass', 'target', 'halo_mass_log', 'halo_mass', 'y']
target_col = None
for col in df_gnn.columns:
    if any(pt in col.strip().lower() for pt in potential_targets):
        target_col = col
        break
if target_col is None:
    target_col = df_gnn.columns[-1] # Fallback to last column
    
print(f"Using '{target_col}' as GNN target.")
y_gnn = df_gnn[target_col].values

# 2. Robust GNN Feature Identification
gnn_cols = [c for c in df_gnn.columns if 'feat' in c.lower() or 'gnn' in c.lower()]
if not gnn_cols:
    # If naming fails, use all columns except the target and known raw columns
    exclude = [target_col, 'group_id', 'is_central']
    gnn_cols = [c for c in df_gnn.columns if c not in exclude]

X_gnn_64 = df_gnn[gnn_cols].values
print(f"Found {len(gnn_cols)} GNN embedding dimensions.")

# 3. PCA Compression for SR efficiency
pca = PCA(n_components=6)
X_gnn_pca = pca.fit_transform(X_gnn_64)
pca_names = [f'pca_{i}' for i in range(6)]
print(f"PCA Explained Variance: {np.sum(pca.explained_variance_ratio_):.2%}")

# ── Feature Sets ─────────────────────────────────────────────
# Notice we add 'y' to the dictionaries to handle the row-count difference
feature_sets = {
    'central_only': {
        'X': np.column_stack([log_M_cen, log_sigma_cen, log_R_cen]),
        'y': y_old,
        'names': ['log_M_cen', 'log_sigma_cen', 'log_R_cen'],
        'desc': 'Properties of the central galaxy only'
    },
    'full_physics': {
        'X': np.column_stack([
            log_M_cen, log_sigma_cen, log_R_cen, 
            log_sigma_sat, log_N_sat, log_M_sat_total
            # VIRIAL PROXY REMOVED
        ]),
        'y': y_old,
        'names': ['log_M_cen', 'log_sigma_cen', 'log_R_cen', 
                  'log_sigma_sat', 'log_N_sat', 'log_M_sat_total'],
        'desc': 'Combined central and satellite kinematics (No Virial Cheat)'
    },
    'gnn_embeddings': {
        'X': X_gnn_pca,
        'y': y_gnn,
        'names': pca_names,
        'desc': 'PCA-reduced GNN latent representations (1593 samples)'
    }
}

# ── Define Custom GP Functions (REQUIRED) ─────────────────────
def _protected_div(x1, x2):
    with np.errstate(divide='ignore', invalid='ignore'):
        return np.where(np.abs(x2) > 1e-6, x1/x2, 1.0)

def _protected_log(x):
    with np.errstate(divide='ignore', invalid='ignore'):
        return np.where(np.abs(x) > 1e-6, np.log10(np.abs(x)), 0.0)

def _protected_sqrt(x):
    return np.sqrt(np.abs(x))

def _cube(x):
    return x ** 3

protected_div  = make_function(function=_protected_div,  name='div',  arity=2)
protected_log  = make_function(function=_protected_log,  name='log',  arity=1)
protected_sqrt = make_function(function=_protected_sqrt, name='sqrt', arity=1)
cube_fn        = make_function(function=_cube,           name='cube', arity=1)

# ── GP Symbolic Regression ────────────────────────────────────
print("\n" + "="*60)
print("GENETIC PROGRAMMING SYMBOLIC REGRESSION")
print(f"Population=15000 | Generations=300 | ~7 hours total")
print("="*60)

all_results = []
start_total = time.time()

for run_name, fset in feature_sets.items():
    X_raw = fset['X']
    y_raw = fset['y']
    names = fset['names']
    desc  = fset['desc']

    print(f"\n{'─'*55}")
    print(f"RUN: {run_name}")
    print(f"Desc: {desc}")
    print(f"Features: {names}")
    print(f"{'─'*55}")

    # Clean NaN/Inf
    mask  = np.all(np.isfinite(X_raw), axis=1) & np.isfinite(y_raw)
    X     = X_raw[mask]
    y_run = y_raw[mask]
    print(f"Clean samples: {X.shape[0]} halos")

    if X.shape[0] < 50:
        print("Too few samples — skipping")
        continue

    t0 = time.time()

    sr = SymbolicRegressor(
        population_size       = 15000,
        generations           = 300,
        tournament_size       = 30,
        stopping_criteria     = 0.000001,
        const_range           = (-20.0, 20.0),
        init_depth            = (2, 6),
        init_method           = 'half and half',
        function_set          = [
            'add', 'sub', 'mul',
            protected_div,
            protected_sqrt,
            protected_log,
            'abs', 'neg',
            cube_fn,
            'sin', 'cos',       # just for fun — might find periodic patterns
        ],
        parsimony_coefficient = 0.004,
        p_crossover           = 0.7,
        p_subtree_mutation    = 0.1,
        p_hoist_mutation      = 0.08,
        p_point_mutation      = 0.1,
        max_samples           = 0.9,
        verbose               = 1,
        random_state          = 42,
        n_jobs                = -1,
        warm_start            = False,
    )

    sr.fit(X, y_run)
    elapsed = time.time() - t0

    y_pred = sr.predict(X)
    mask2  = np.isfinite(y_pred)
    y_pred = y_pred[mask2]
    y_eval = y_run[mask2]

    rmse  = np.sqrt(np.mean((y_eval - y_pred)**2))
    r2    = 1 - np.sum((y_eval-y_pred)**2) / \
                np.sum((y_eval-y_eval.mean())**2)
    mae   = np.mean(np.abs(y_eval - y_pred))

    eq_raw   = str(sr._program)
    eq_named = eq_raw
    # Reverse iteration prevents variable name overwriting (e.g. X1 vs X10)
    for i, name in reversed(list(enumerate(names))):
        eq_named = eq_named.replace(f'X{i}', name)

    # Physics pattern detection
    uses_sigma  = any(s in eq_named for s in ['log_sigma','virial','fj_proxy'])
    uses_radius = any(s in eq_named for s in ['log_R','log_extent','log_M_dyn'])
    uses_vel    = 'log_vel_extent' in eq_named
    uses_n      = 'log_N' in eq_named
    uses_trig   = any(s in eq_raw for s in ['sin','cos'])

    if uses_sigma and uses_radius:
        pattern = "VIRIAL-LIKE: M ~ f(sigma, R)"
    elif uses_sigma and not uses_radius:
        pattern = "FABER-JACKSON-LIKE: M ~ f(sigma)"
    elif uses_radius and not uses_sigma:
        pattern = "SIZE SCALING: M ~ f(R)"
    elif uses_trig:
        pattern = "OSCILLATORY — unexpected, investigate"
    else:
        pattern = "COMPOSITE"

    print(f"\n{'='*40}")
    print(f"RESULT: {run_name}")
    print(f"  Equation : {eq_named}")
    print(f"  RMSE     : {rmse:.4f} dex")
    print(f"  R²       : {r2:.4f}")
    print(f"  MAE      : {mae:.4f} dex")
    print(f"  Nodes    : {sr._program.length_}")
    print(f"  Depth    : {sr._program.depth_}")
    print(f"  Pattern  : {pattern}")
    print(f"  Runtime  : {elapsed/60:.1f} mins")
    print(f"  Trig terms: {uses_trig}")

    result = {
        'run'        : run_name,
        'desc'       : desc,
        'eq_raw'     : eq_raw,
        'equation'   : eq_named,
        'rmse'       : float(rmse),
        'r2'         : float(r2),
        'mae'        : float(mae),
        'tree_size'  : int(sr._program.length_),
        'depth'      : int(sr._program.depth_),
        'pattern'    : pattern,
        'runtime_min': float(elapsed/60),
        'n_samples'  : int(X.shape[0]),
        'features'   : names,
        'uses_trig'  : uses_trig,
    }
    all_results.append(result)

    # Save checkpoint after each run
    os.makedirs('/kaggle/working/outputs/equations', exist_ok=True)
    with open(f'/kaggle/working/outputs/equations/{run_name}_result.json','w') as f:
        json.dump(result, f, indent=2)
    print(f"  Checkpoint saved")

total_time = (time.time() - start_total) / 60
print(f"\nTotal runtime: {total_time:.1f} mins")

# ── Linear Baseline ───────────────────────────────────────────
print("\n" + "="*60)
print("LINEAR REGRESSION BASELINE")
print("="*60)

linear_results = []
for run_name, fset in feature_sets.items():
    X_raw = fset['X']
    y_raw = fset['y']
    mask  = np.all(np.isfinite(X_raw), axis=1) & np.isfinite(y_raw)
    Xc    = X_raw[mask]
    yc    = y_raw[mask]

    if Xc.shape[0] < 20:
        continue

    scaler = StandardScaler()
    Xsc    = scaler.fit_transform(Xc)
    lr     = LinearRegression().fit(Xsc, yc)
    yp     = lr.predict(Xsc)
    r2_lr  = 1 - np.sum((yc-yp)**2)/np.sum((yc-yc.mean())**2)
    rmse_lr = np.sqrt(np.mean((yc-yp)**2))

    # Cross-validated R²
    cv_r2  = cross_val_score(LinearRegression(), Xsc, yc,
                              cv=5, scoring='r2').mean()

    coefs  = " + ".join([f"{c:+.3f}*{n}"
                          for c,n in zip(lr.coef_, fset['names'])])
    eq_str = f"{lr.intercept_:.3f} {coefs}"

    print(f"\n{run_name}:")
    print(f"  R²     = {r2_lr:.4f} (CV R² = {cv_r2:.4f})")
    print(f"  RMSE   = {rmse_lr:.4f} dex")
    print(f"  Eq     : {eq_str}")

    linear_results.append({
        'run': run_name, 'r2': r2_lr, 'cv_r2': cv_r2,
        'rmse': rmse_lr, 'equation': eq_str
    })

# ── Final Summary ─────────────────────────────────────────────
print("\n" + "="*60)
print("FINAL SUMMARY — RANKED BY R²")
print("="*60)

all_results.sort(key=lambda x: x['r2'], reverse=True)
for i, r in enumerate(all_results):
    print(f"\n#{i+1} [{r['run']}] R²={r['r2']:.4f} | "
          f"RMSE={r['rmse']:.4f} | {r['pattern']}")
    print(f"   {r['equation']}")

if all_results:
    best = all_results[0]
    print(f"\n{'='*60}")
    print(f"BEST EQUATION")
    print(f"{'='*60}")
    print(f"  {best['equation']}")
    print(f"  R²={best['r2']:.4f} | RMSE={best['rmse']:.4f} dex | "
          f"{best['tree_size']} nodes")
    print(f"  {best['pattern']}")

    # Physical constants check
    eq_str = best['equation']
    print(f"\n  Physical constants detected in equation:")
    for val, name in [
        (str(round(LOG_G,2)),    'log(G) gravitational'),
        (str(round(LOG_PI,2)),   'log(pi)'),
        (str(round(LOG_E,2)),    'log(e) Euler'),
        (str(round(LOG_C,2)),    'log(c) light speed'),
        (str(round(LOG_H100,2)), 'log(h) Hubble param'),
    ]:
        if val in eq_str:
            print(f"    Found: {name} = {val}")

# ── Save Everything ───────────────────────────────────────────
os.makedirs('/kaggle/working/outputs/equations', exist_ok=True)

with open('/kaggle/working/outputs/equations/sr_all_results.json','w') as f:
    json.dump(all_results, f, indent=2)

pd.DataFrame(all_results).to_csv(
    '/kaggle/working/outputs/equations/sr_summary.csv', index=False)

pd.DataFrame(linear_results).to_csv(
    '/kaggle/working/outputs/equations/linear_baselines.csv', index=False)

if all_results:
    best = all_results[0]
    with open('/kaggle/working/outputs/equations/best_equation_paper.txt','w') as f:
        f.write("Cosmic-Net: Best Discovered Symbolic Equation\n")
        f.write("="*50 + "\n\n")
        f.write(f"Equation   : {best['equation']}\n")
        f.write(f"R²         : {best['r2']:.4f}\n")
        f.write(f"RMSE       : {best['rmse']:.4f} dex\n")
        f.write(f"MAE        : {best['mae']:.4f} dex\n")
        f.write(f"Complexity : {best['tree_size']} tree nodes\n")
        f.write(f"Pattern    : {best['pattern']}\n\n")
        f.write("NOTE: Features are satellite-only (leak-free)\n")
        f.write("Target: halo_mass_log (requires re-fetch validation)\n\n")
        f.write("All runs:\n")
        for r in all_results:
            f.write(f"  [{r['run']}] R²={r['r2']:.4f} RMSE={r['rmse']:.4f} "
                    f"| {r['equation']}\n")

print("\nSaved:")
print("  sr_all_results.json")
print("  sr_summary.csv")
print("  linear_baselines.csv")
print("  best_equation_paper.txt")
print("  {run_name}_result.json  (one per run, checkpointed)")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 85.0 MB/s eta 0:00:00
Dependencies installed

Physical constants (log10):
  log10(G)    = -8.3663   [Mpc (km/s)^2 / M_sun]
  log10(pi)   = 0.4971
  log10(e)    = 0.4343
  log10(c)    = 5.4768   [km/s]
  log10(h)    = -0.1692  [Hubble param]

Loading data...

Processing Centrals and Satellites...

Loading GNN Embedding data...
Using 'log_metallicity' as GNN target.
Found 64 GNN embedding dimensions.
PCA Explained Variance: 99.73%

GENETIC PROGRAMMING SYMBOLIC REGRESSION
Population=15000 | Generations=300 | ~7 hours total

───────────────────────────────────────────────────────
RUN: central_only
Desc: Properties of the central galaxy only
Features: ['log_M_cen', 'log_sigma_cen', 'log_R_cen']
───────────────────────────────────────────────────────
Clean samples: 541 halos
    |   Population Average    |             Best Individual              |
---- -------

/tmp/ipykernel_17/2744216339.py:176: RuntimeWarning: overflow encountered in power
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:573: RuntimeWarning: invalid value encountered in multiply
  avg = avg_as_array = np.multiply(a, wgt,


  49    10.61              inf       11        0.0463469        0.0865448    184.31m
  50    10.53      3.26454e+40       11        0.0464375        0.0857438    188.41m
  51    10.64      1.22561e+57       11        0.0465245        0.0849752    185.89m
  52    10.69       2.0011e+40       11        0.0458525        0.0909127    180.57m
  53    10.59      3.34221e+40       11        0.0464397        0.0857243    178.96m
  54    10.66      2.66532e+47       11        0.0467604        0.0828909    172.03m
  55    10.60      7.27719e+39       11        0.0468381        0.0822042    178.44m
  56    10.68      2.18823e+40       11        0.0461618          0.08818    188.29m
  57    10.62       2.1889e+40       11         0.045241        0.0963168    188.47m
  58    10.61      3.58559e+50       11        0.0464504        0.0856299    177.23m
  59    10.64      1.36941e+57       11        0.0460646        0.0890386    179.48m
  60    10.63      3.66798e+40       11        0.0459597        0